# Bài 13: Đạo hàm bậc hai và Toán tử Laplace (Second Derivatives and the Laplacian)

Bài 12 tìm biên dưới dạng các *đỉnh* của đạo hàm bậc nhất (độ lớn gradient). Bài này dùng đạo hàm **bậc hai** thay vào đó, nơi biên xuất hiện dưới dạng các **điểm cắt không (zero crossings)** &mdash; nền tảng của bộ phát hiện biên Marr-Hildreth kinh điển. Sau đó ta dùng lại cùng ý tưởng làm-mờ-và-lấy-hiệu để xây dựng **kim tự tháp Laplace (Laplacian pyramid)**, khắc phục vấn đề mất thông tin của kim tự tháp Gauss ở Bài 11.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Tại sao lại là điểm cắt không? Trực giác 1D (Why zero crossings? A 1D intuition)
Hãy xét một biên bậc duy nhất dọc theo một hàng. Đạo hàm bậc nhất là một gai nhọn tại biên; đạo hàm bậc hai dao động từ dương sang âm (hoặc ngược lại) và cắt qua đúng giá trị 0 ngay tại vị trí biên.

In [ ]:
x = np.linspace(0, 10, 400)
step = 1 / (1 + np.exp(-6 * (x - 5)))  # a smoothed step, to keep derivatives well-defined
first_deriv = np.gradient(step, x)
second_deriv = np.gradient(first_deriv, x)

fig, axes = plt.subplots(3, 1, figsize=(6, 6), sharex=True)
for ax, y, title in zip(axes, [step, first_deriv, second_deriv],
                         ['Intensity (step edge)', 'First derivative (peak at edge)', 'Second derivative (zero crossing at edge)']):
    ax.plot(x, y)
    ax.axvline(5, color='gray', linestyle='--', linewidth=1)
    ax.set_title(title, fontsize=10)
if True:
    axes[2].axhline(0, color='red', linewidth=0.8)
plt.tight_layout()
plt.show()

## Toán tử Laplace: đạo hàm bậc hai trong 2D (The Laplacian: a 2D second derivative)
Toán tử Laplace (Laplacian) cộng các đạo hàm riêng bậc hai không hỗn hợp của một ảnh 2D:

$$\nabla^2 I = \frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2}$$

Khác với gradient, nó là một giá trị vô hướng duy nhất tại mỗi điểm ảnh (không có hướng) &mdash; nó chỉ đo mức độ một điểm ảnh khác biệt so với trung bình các điểm lân cận. Hai kernel rời rạc phổ biến:

$$K_4 = \begin{bmatrix}0&1&0\\1&-4&1\\0&1&0\end{bmatrix} \qquad K_8 = \begin{bmatrix}1&1&1\\1&-8&1\\1&1&1\end{bmatrix}$$

$K_4$ chỉ dùng các lân cận liên thông 4; $K_8$ bao gồm cả các đường chéo để có phản hồi mạnh hơn, đẳng hướng (isotropic) hơn.

In [ ]:
img = np.zeros((150, 150), dtype=np.float64)
cv2.rectangle(img, (30, 30), (100, 100), 200, -1)
cv2.circle(img, (110, 110), 25, 120, -1)

k4 = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float64)
lap = cv2.filter2D(img, cv2.CV_64F, k4)

print('matches cv2.Laplacian?', np.allclose(lap, cv2.Laplacian(img, cv2.CV_64F, ksize=1)))

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original')
axes[1].imshow(lap, cmap='coolwarm', vmin=-200, vmax=200)
axes[1].set_title('Laplacian response\n(red=positive, blue=negative)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Mỗi biên tạo ra một thùy dương ở một bên và một thùy âm ở bên kia &mdash; bản thân biên nằm tại điểm cắt không giữa chúng.

## Bộ phát hiện Marr-Hildreth: Laplace của Gauss (LoG) (The Marr-Hildreth detector: Laplacian of Gaussian (LoG))
Toán tử Laplace, giống như mọi phép đạo hàm, khuếch đại nhiễu tần số cao &mdash; và đạo hàm *bậc hai* còn khuếch đại nhiễu mạnh hơn cả đạo hàm bậc nhất. <a href="../references.html#marr-hildreth-1980">Cách khắc phục của Marr và Hildreth (1980)</a>: **làm trơn trước**, rồi mới lấy Laplace. Vì tích chập có tính kết hợp, điều này tương đương với tích chập trực tiếp với một kernel kết hợp duy nhất, gọi là **Laplace của Gauss (Laplacian of Gaussian — LoG)**:

$$\text{LoG}_\sigma(I) = \nabla^2(G_\sigma * I) = (\nabla^2 G_\sigma) * I$$

Sau đó biên được tìm dưới dạng các **điểm cắt không** của phản hồi LoG, thay vì phân ngưỡng trực tiếp độ lớn của nó.

In [ ]:
rng = np.random.default_rng(1)
disk = np.zeros((150, 150), dtype=np.float64)
cv2.circle(disk, (75, 75), 50, 255, -1)
noisy_disk = np.clip(disk + rng.normal(0, 10, disk.shape), 0, 255)

sigma = 2.0
log_response = cv2.Laplacian(cv2.GaussianBlur(noisy_disk, (0, 0), sigmaX=sigma), cv2.CV_64F, ksize=3)


def zero_crossings(response, threshold=4.0):
    """Mark a pixel as an edge if it's adjacent to a sign change of at least `threshold` magnitude."""
    edges = np.zeros_like(response, dtype=np.uint8)
    shifts = [(0, 1), (1, 0), (1, 1), (1, -1)]
    for dy, dx in shifts:
        shifted = np.roll(np.roll(response, dy, axis=0), dx, axis=1)
        sign_change = (response * shifted < 0) & (np.abs(response - shifted) > threshold)
        edges |= sign_change.astype(np.uint8)
    return edges * 255

log_edges = zero_crossings(log_response)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, im, title in zip(axes, [noisy_disk, log_response, log_edges],
                          ['Noisy input', 'LoG response', 'Zero crossings\n(Marr-Hildreth edges)']):
    ax.imshow(im, cmap='gray' if title != 'LoG response' else 'coolwarm')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Phép xấp xỉ rẻ hơn: Hiệu của Gauss (DoG) (The cheaper approximation: Difference of Gaussians (DoG))
Tính một kernel LoG thực sự tốn kém hơn làm mờ. Một lối tắt được dùng rộng rãi: *hiệu* của hai lần làm mờ Gauss ở các tỉ lệ hơi khác nhau xấp xỉ một LoG đã nhân hệ số:

$$\text{DoG} = G_{\sigma} - G_{k\sigma} \;\approx\; -(k-1)\sigma^2 \cdot \text{LoG}_\sigma$$

Đây cũng là khối xây dựng mà sau này SIFT dùng lại để phát hiện điểm đặc trưng (keypoint) ở nhiều tỉ lệ.

In [ ]:
k = 1.6
g_sigma = cv2.GaussianBlur(noisy_disk, (0, 0), sigmaX=sigma)
g_ksigma = cv2.GaussianBlur(noisy_disk, (0, 0), sigmaX=sigma * k)
dog = (g_ksigma - g_sigma) / ((k - 1) * sigma**2)

correlation = np.corrcoef(log_response.ravel(), dog.ravel())[0, 1]
print(f'correlation between LoG and scaled DoG: {correlation:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(log_response, cmap='coolwarm', vmin=-30, vmax=30)
axes[0].set_title('LoG')
axes[1].imshow(dog, cmap='coolwarm', vmin=-30, vmax=30)
axes[1].set_title('DoG (scaled)')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Hai kết quả có độ tương quan cao nhưng không giống hệt nhau &mdash; DoG là một phép xấp xỉ, không phải sự thay thế chính xác, nhưng rẻ hơn nhiều (hai lần làm mờ và một phép trừ so với một kernel đạo hàm bậc hai tường minh).

## Kim tự tháp Laplace: khôi phục những gì kim tự tháp Gauss vứt bỏ (Laplacian pyramids: recovering what Gaussian pyramids throw away)
Bài 11 cho thấy kim tự tháp Gauss làm mất thông tin: làm mờ và giảm mẫu lặp đi lặp lại, rồi cố tăng mẫu trở lại, không tái tạo được ảnh gốc. **Kim tự tháp Laplace** khắc phục điều này bằng cách lưu, ở mỗi tầng, chính xác phần chi tiết mà nếu không sẽ bị mất &mdash; tức *hiệu* giữa một tầng Gauss và tầng thô hơn được tăng mẫu trở lại để khớp với nó. Hiệu đó là một phép xấp xỉ rời rạc của toán tử Laplace, vì thế mới có cùng tên gọi.

In [ ]:
def gaussian_pyramid(image, num_levels):
    pyramid = [image]
    current = image
    for _ in range(num_levels - 1):
        current = cv2.pyrDown(current)
        pyramid.append(current)
    return pyramid


def laplacian_pyramid(gauss_pyramid):
    lap_pyramid = []
    for i in range(len(gauss_pyramid) - 1):
        size = (gauss_pyramid[i].shape[1], gauss_pyramid[i].shape[0])
        upsampled = cv2.pyrUp(gauss_pyramid[i + 1], dstsize=size)
        detail = gauss_pyramid[i].astype(np.int16) - upsampled.astype(np.int16)
        lap_pyramid.append(detail)
    lap_pyramid.append(gauss_pyramid[-1].astype(np.int16))  # the smallest level: no finer detail to subtract
    return lap_pyramid


def reconstruct_from_laplacian(lap_pyramid):
    current = lap_pyramid[-1]
    for i in range(len(lap_pyramid) - 2, -1, -1):
        size = (lap_pyramid[i].shape[1], lap_pyramid[i].shape[0])
        upsampled = cv2.pyrUp(current, dstsize=size)
        current = upsampled + lap_pyramid[i]
    return np.clip(current, 0, 255).astype(np.uint8)


photo = np.zeros((256, 256, 3), dtype=np.uint8)
photo[:] = (40, 40, 40)
cv2.circle(photo, (128, 128), 90, (255, 120, 30), -1)
cv2.rectangle(photo, (30, 30), (110, 110), (30, 200, 255), -1)

g_pyr = gaussian_pyramid(photo, num_levels=5)
l_pyr = laplacian_pyramid(g_pyr)

fig, axes = plt.subplots(1, len(l_pyr), figsize=(13, 3))
for ax, level in zip(axes, l_pyr):
    display = np.clip(level.astype(np.int32) + 128, 0, 255).astype(np.uint8)  # shift for visibility
    ax.imshow(display)
    ax.set_title(f'{level.shape[1]}x{level.shape[0]}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Mỗi tầng (được hiển thị dịch lên +128 mức xám để thấy được các giá trị âm) phần lớn là màu xám phẳng, ngoại trừ ngay tại các biên &mdash; chính là nơi chi tiết bị mất do làm mờ và giảm mẫu. Tầng cuối cùng, nhỏ nhất lưu nội dung ảnh thực sự chứ không phải một hiệu, vì không còn tầng nào thô hơn để so sánh.

### Tái tạo chính xác (Exact reconstruction)
Vì mỗi tầng lưu chính xác những gì tầng tương ứng của kim tự tháp Gauss đã loại bỏ, cộng ngược lên theo kim tự tháp sẽ tái tạo ảnh gốc một cách chính xác (sai khác do làm tròn số nguyên).

In [ ]:
reconstructed = reconstruct_from_laplacian(l_pyr)
diff = np.abs(reconstructed.astype(int) - photo.astype(int))

print(f'max reconstruction error  = {diff.max()} gray levels')
print(f'mean reconstruction error = {diff.mean():.4f} gray levels')

fig, axes = plt.subplots(1, 2, figsize=(6, 3.5))
axes[0].imshow(photo)
axes[0].set_title('Original')
axes[1].imshow(reconstructed)
axes[1].set_title('Reconstructed from Laplacian pyramid')
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

Hãy so sánh với kết quả `pyrDown` + `pyrUp` ở Bài 11, vốn có sai số tái tạo khác không thấy rõ: kim tự tháp Laplace lưu vừa đủ thông tin bổ sung ở mỗi tầng để làm cho quá trình hoàn toàn khả nghịch, với cái giá là phải giữ lại tất cả các tầng (không chỉ tầng nhỏ nhất).

### Bài tập (Exercises)
1. Hãy thử `threshold=1.0` và `threshold=15.0` trong `zero_crossings`. Số điểm ảnh biên phát hiện được thay đổi thế nào, và tại sao một ngưỡng *cao hơn* trên tiêu chí đổi dấu của đạo hàm *bậc hai* lại hoạt động khác với một ngưỡng độ lớn trên đạo hàm bậc nhất?
2. Tăng mức nhiễu trong `noisy_disk` và xem `sigma` cần lớn đến đâu thì bản đồ biên điểm cắt không mới hết bị chi phối bởi các vòng nhiễu giả.
3. Kim tự tháp Laplace là công cụ kinh điển đằng sau phép trộn ảnh liền mạch (seamless blending) (ví dụ trộn hai bức ảnh dọc theo một mặt nạ, từng tầng một). Hãy phác thảo &mdash; bằng lời hoặc bằng code &mdash; cách bạn trộn hai kim tự tháp Laplace (ví dụ lấy trung bình chúng với trọng số thay đổi theo không gian ở mỗi tầng) trước khi tái tạo, và tại sao trộn trong biểu diễn này lại tránh được các đường nối sắc cạnh mà trộn trực tiếp các ảnh gốc sẽ tạo ra.